# 🔬 Notebook 4 — Transfer Learning
## Aerial Bird vs Drone Classification

**Models:** ResNet50 · MobileNetV2 · EfficientNetB0

**Training Strategy:**
- Phase 1: Freeze backbone → train head only
- Phase 2: Unfreeze top N layers → fine-tune

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(ROOT))

from src.utils import set_seed, ensure_dirs
from src.config import PLOTS_DIR, SAVED_MODELS_DIR
from src.data_preprocessing import get_all_generators
from src.transfer_learning import (
    build_resnet50, build_mobilenet, build_efficientnet,
    train_transfer_model, train_all_transfer_models
)
from src.evaluate import evaluate_model, compare_models

set_seed()
ensure_dirs(PLOTS_DIR, SAVED_MODELS_DIR)
train_gen, valid_gen, test_gen = get_all_generators()
print('✅ Ready for Transfer Learning')

## 1. Model Architecture Preview

In [ ]:
# Preview MobileNetV2 (fastest to inspect)
model_preview = build_mobilenet()
print('MobileNetV2 Total Params:', f'{model_preview.count_params():,}')
model_preview.summary()

## 2. Train MobileNetV2 (Fast / Recommended)

In [ ]:
mobilenet_model, h1, h2 = train_transfer_model(
    'mobilenet', train_gen, valid_gen,
    epochs_phase1=15,
    epochs_phase2=20
)

## 3. Train ResNet50

In [ ]:
resnet_model, h1, h2 = train_transfer_model(
    'resnet50', train_gen, valid_gen,
    epochs_phase1=15,
    epochs_phase2=20
)

## 4. Train EfficientNetB0

In [ ]:
effnet_model, h1, h2 = train_transfer_model(
    'efficientnet', train_gen, valid_gen,
    epochs_phase1=15,
    epochs_phase2=20
)

## 5. Evaluate All on Test Set

In [ ]:
model_objects = {
    'MobileNetV2':    mobilenet_model,
    'ResNet50':       resnet_model,
    'EfficientNetB0': effnet_model,
}

all_results = {}
for name, mdl in model_objects.items():
    test_gen.reset()
    res = evaluate_model(mdl, test_gen, model_name=name)
    all_results[name] = res

df = compare_models(all_results)
df